# 06 · Product groups and the listing in force

**The question.** This is S1.1 in [`SYSTEM CODE/DESIGN_QUESTIONS.md`](../DESIGN_QUESTIONS.md). The rule for product groups is already set in [`DATA/ALIGNING_LEMIS_AND_CITES.md`](../../DATA/ALIGNING_LEMIS_AND_CITES.md) §2.1: groups are defined by how each vocabulary records an object, not by matching names. The current pipeline matched names (ALIGNING §3, #14). This notebook applies the rule and records the evidence for each choice.

**Decision rule (written in DESIGN_QUESTIONS before this notebook was run):**
1. No group holds at least 0.1% of one dataset and nothing in the other.
2. Every group whose shares differ more than tenfold is explained as either a **coding difference** (fixed in the crosswalk) or a **difference in trade** (kept, and stated).
3. Caviar, extracts, ivory, horn and bone stay apart (notebook 01, L9: their case rates differ widely).

**Populations.** LEMIS: the training population (animals with a class, 2000–2010). CITES: exporter-reported records 2015–2024, the population for counts and shares (ALIGNING §1.2).

**Telling coding from trade.** LEMIS records every US wildlife import, CITES only listed species. So a gap can simply mean the US imports a lot of something CITES does not cover. Each comparison is therefore repeated with LEMIS restricted to **species with a CITES listing in force**. A gap that closes under that restriction is a difference in what is traded; a gap that stays needs a look at how each vocabulary codes the object.

The notebook compares the aligned files before this change (kept in `DATA/processed/pre_rework/`) with the rebuilt ones.

**A second finding.** While separating coding from trade, the listing used for the restriction turned out to be wrong for one kind of record. Section 6 shows the evidence and the change to the appendix rule.

In [1]:
import duckdb
import matplotlib.pyplot as plt
import pandas as pd

import eda_setup as E
from wildguard import config
from wildguard.schema_alignment import NON_ANIMAL_CLASSES

E.style()
con = E.connect()          # raw CITES views (cites_animals), for the listing in force in section 6
OLD = config.PROCESSED_DIR / "pre_rework"
non_animal = "(" + ", ".join(f"'{c}'" for c in sorted(NON_ANIMAL_CLASSES)) + ")"
con.execute(f"""
    CREATE VIEW lemis_old AS SELECT * FROM '{OLD / "lemis_aligned.parquet"}'
        WHERE class IS NOT NULL AND class NOT IN {non_animal} AND year <= 2010;
    CREATE VIEW cites_old AS SELECT * FROM '{OLD / "cites_aligned.parquet"}'
        WHERE year BETWEEN 2015 AND 2024 AND reporter_type = 'E' AND appendix <> 'N';
    CREATE VIEW lemis_new AS SELECT * FROM '{config.LEMIS_PARQUET}' WHERE animal AND year <= 2010;
    CREATE VIEW cites_new AS SELECT * FROM '{config.CITES_PARQUET}'
        WHERE year BETWEEN 2015 AND 2024 AND reporter_type = 'E';
""")


def Q(sql):
    return con.execute(sql).df()


def shares(lemis, cites, listed=None):
    """Product-group shares in LEMIS and CITES; `listed` restricts LEMIS to listed species."""
    extra = f"WHERE {listed}" if listed else ""
    t = Q(f"""
        WITH l AS (SELECT product_group AS g, count(*) AS n FROM {lemis} {extra} GROUP BY 1),
             c AS (SELECT product_group AS g, count(*) AS n FROM {cites} GROUP BY 1)
        SELECT coalesce(l.g, c.g) AS product_group,
               coalesce(l.n, 0) / (SELECT sum(n) FROM l) AS lemis,
               coalesce(c.n, 0) / (SELECT sum(n) FROM c) AS cites
        FROM l FULL JOIN c ON l.g = c.g""").set_index("product_group")
    t["ratio"] = t.max(axis=1) / t[["lemis", "cites"]].min(axis=1)
    return t.sort_values("lemis", ascending=False)


def flag(t):
    one_sided = (t[["lemis", "cites"]].max(axis=1) >= 0.001) & (t[["lemis", "cites"]].min(axis=1) == 0)
    return t.assign(one_sided=one_sided, over_10x=(t.ratio > 10) & ~one_sided)

## 1. Before: groups matched by name

In [2]:
before = flag(shares("lemis_old", "cites_old"))
before.style.format({"lemis": "{:.2%}", "cites": "{:.2%}", "ratio": "{:.1f}"}).apply(
    lambda r: ["background-color: #f6d6c8" if (r.one_sided or r.over_10x) else "" for _ in r], axis=1)

,lemis,cites,ratio,one_sided,over_10x
product_group,,,,,
live,28.32%,22.57%,1.3,False,False
trophies,19.46%,1.88%,10.4,False,True
leather_small,10.52%,56.38%,5.4,False,False
shells,10.20%,0.03%,298.0,False,True
carvings_jewellery,4.90%,0.70%,7.0,False,False
garments,4.52%,1.45%,3.1,False,False
shoes,4.35%,0.00%,inf,True,False
skins,2.53%,4.04%,1.6,False,False
specimens,2.20%,3.13%,1.4,False,False


Two groups break rule 1: **shoes** and **coral_live_rock** hold 4.3% and 0.5% of LEMIS and nothing in CITES. Several more differ more than tenfold.

## 2. The two one-sided groups

**Shoes.** What LEMIS shoes are made of, and how CITES records the same kind of object:

In [3]:
display(Q("""SELECT class, count(*) AS line_items, avg(sp_listed::INT) AS share_species_listed
             FROM lemis_new WHERE product_code = 'SHO' GROUP BY 1 ORDER BY 2 DESC LIMIT 4"""))
Q("""SELECT product_code AS cites_term, count(*) AS records,
            count(*) FILTER (WHERE class = 'Reptilia') AS reptile_records
     FROM cites_new WHERE product_code IN ('shoes', 'leather products (small)', 'leather products (large)',
                                       'garments')
     GROUP BY 1 ORDER BY 2 DESC""")

,class,line_items,share_species_listed
0,Reptilia,107697,0.5666
1,Mammalia,21253,0.08949
2,Aves,15696,0.9808
3,Actinopteri,4761,0.03109


,cites_term,records,reptile_records
0,leather products (small),2726005,2713814
1,garments,70168,41126
2,leather products (large),13753,11912


LEMIS shoes are mostly reptile leather. CITES has a term for shoes but no exporter used it in 2015–2024; reptile leather goods are recorded as *leather products (small)*. **Coding difference:** LEMIS `SHO` joins the leather group.

**Live rock.** LEMIS has a code for live rock (`ROC`); CITES has no such term. How CITES records the same coral:

In [4]:
display(Q("SELECT genus, count(*) AS line_items FROM lemis_new WHERE product_code = 'ROC' GROUP BY 1 ORDER BY 2 DESC LIMIT 3"))
Q("""SELECT product_code AS cites_term, unit_group, count(*) AS records FROM cites_new
     WHERE genus = 'scleractinia' GROUP BY 1, 2 ORDER BY 3 DESC LIMIT 5""")

,genus,line_items
0,scleractinia,17672
1,agaricia,14
2,acropora,12


,cites_term,unit_group,records
0,raw corals,count,19619
1,raw corals,kg,13991
2,live,count,1110
3,live,kg,627
4,specimens,count,101


LEMIS live rock is almost all *Scleractinia*, which CITES records as *raw corals*. **Coding difference:** `ROC` joins `coral_raw`.

## 3. Leather goods: small, large and shoes

LEMIS splits manufactured leather goods three ways; CITES nearly always uses one term. If the split carried risk information it would be worth keeping on the LEMIS side; the case rates say it does not:

In [5]:
display(Q("""SELECT product_code, count(*) AS line_items, avg(label) AS case_rate FROM lemis_new
             WHERE product_code IN ('LPS', 'LPL', 'SHO', 'TRI', 'GAR') GROUP BY 1 ORDER BY 2 DESC""")
        .style.format({"case_rate": "{:.2%}"}))
Q("""SELECT product_code AS cites_term, count(*) AS records FROM cites_new
     WHERE product_code LIKE '%leather%' OR product_code LIKE 'fur%' GROUP BY 1 ORDER BY 2 DESC""")

,product_code,line_items,case_rate
0,LPS,232361,3.02%
1,GAR,161127,0.56%
2,SHO,154893,3.14%
3,TRI,142338,0.81%
4,LPL,71252,2.03%


,cites_term,records
0,leather products (small),2726005
1,leather products (large),13753
2,fur product (small),312
3,fur products (large),139


Small goods, large goods and shoes are enforced at similar rates (2.0–3.1%), and CITES uses *leather products (large)* on 0.5% of leather records. **Coding difference:** one group, `leather_products`, for small and large goods and shoes, and for the matching CITES terms. Trim (`TRI`) stays in the group, as it was before: its case rate is lower (0.8%), but CITES has no term for trim. Garments stay separate: both vocabularies have them, and their case rate (0.6%) is far below that of leather goods.

## 4. Codes outside the crosswalk

In [6]:
display(Q(f"""SELECT o.product_code, count(*) AS line_items FROM lemis_old o
              WHERE o.product_group = 'other' AND o.product_code NOT IN ('UNS')
              GROUP BY 1 ORDER BY 2 DESC LIMIT 12"""))
Q("""SELECT product_code AS cites_term, count(*) AS records FROM cites_old
     WHERE product_group = 'other' AND product_code <> 'unspecified' GROUP BY 1 ORDER BY 2 DESC""")

,product_code,line_items
0,WPR,171
1,VEN,69
2,CUL,52
3,DPL,36
4,BAR,25
5,LVS,13
6,TIM,11
7,GRS,5
8,SEE,5
9,STE,3


,cites_term,records
0,fibres,320
1,thread,6
2,wood product,5
3,cultures,5


Read with the LEMIS code list (`DATA/reference/lemis_codes.csv`), the LEMIS codes are plant and wood products: `VEN` is wood **veneer** (not venom) and `FIB` is **plant fibre**. They stay in `other`. CITES *fibres* on animal records are animal fibre (for example vicuña wool), so they join `hair`. The plant terms on a few CITES animal records stay in `other`.

## 5. After: groups defined by coding

In [7]:
after = flag(shares("lemis_new", "cites_new"))
after_listed = shares("lemis_new", "cites_new", listed="sp_listed")
after = after.join(after_listed[["lemis", "ratio"]].rename(columns={"lemis": "lemis_listed_species",
                                                                   "ratio": "ratio_listed_species"}))
after.style.format({"lemis": "{:.2%}", "cites": "{:.2%}", "lemis_listed_species": "{:.2%}",
                    "ratio": "{:.1f}", "ratio_listed_species": "{:.1f}"}).apply(
    lambda r: ["background-color: #f6d6c8" if (r.one_sided or r.over_10x) else "" for _ in r], axis=1)

,lemis,cites,ratio,one_sided,over_10x,lemis_listed_species,ratio_listed_species
product_group,,,,,,,
live,28.32%,22.57%,1.3,False,False,32.25%,1.4
trophies,19.46%,1.88%,10.4,False,True,18.22%,9.7
leather_products,16.87%,56.67%,3.4,False,False,33.80%,1.7
shells,10.20%,0.03%,298.0,False,True,0.31%,9.0
carvings_jewellery,4.90%,0.70%,7.0,False,False,0.25%,2.8
garments,4.52%,1.45%,3.1,False,False,1.36%,1.1
skins,2.53%,4.04%,1.6,False,False,3.64%,1.1
specimens,2.20%,3.13%,1.4,False,False,1.93%,1.6
meat,1.95%,0.58%,3.4,False,False,0.94%,1.6


No group is one-sided any more (rule 1). The groups that still differ more than tenfold need rule 2's explanation:

In [8]:
GAPS = after[after.over_10x].index.tolist()
rows = []
for g in GAPS:
    top = Q(f"""SELECT genus, count(*) AS n FROM lemis_new WHERE product_group = '{g}'
                GROUP BY 1 ORDER BY 2 DESC LIMIT 3""")
    purpose = Q(f"""SELECT purpose, count(*) AS n FROM lemis_new WHERE product_group = '{g}'
                    GROUP BY 1 ORDER BY 2 DESC LIMIT 1""")
    listed = Q(f"SELECT avg(sp_listed::INT) AS s FROM lemis_new WHERE product_group = '{g}'").s[0]
    rows.append({"group": g, "LEMIS top genera": ", ".join(top.genus.fillna("(unknown)")),
                 "LEMIS main purpose": purpose.purpose[0],
                 "LEMIS share species-listed": listed})
pd.DataFrame(rows).set_index("group").style.format({"LEMIS share species-listed": "{:.1%}"})

,LEMIS top genera,LEMIS main purpose,LEMIS share species-listed
group,,,
trophies,"ursus, tragelaphus, aepyceros",H,24.6%
shells,"pinctada, tectus, trochus",T,0.8%
horn,"odocoileus, cervus, alces",T,2.4%
hair,"odocoileus, cervus, meles",T,5.5%
other,"(unknown), sebastes, collocalia",T,4.8%
caviar,"acipenser, huso, acipenseriformes",T,80.4%
scales,"linckia, diodon, protoreaster",T,4.5%
cloth,"vicugna, mustela, vulpes",T,38.6%
fins,"elasmobranchii, halocynthia, oncorhynchus",S,5.4%


In [9]:
display(Q("""SELECT product_code AS cites_term, count(*) AS records FROM cites_new
             WHERE genus IN ('strombus', 'lobatus') GROUP BY 1 ORDER BY 2 DESC LIMIT 4"""))
display(Q("""SELECT product_code AS cites_term, count(*) AS records FROM cites_new WHERE purpose = 'H'
             GROUP BY 1 ORDER BY 2 DESC LIMIT 5"""))
Q("""SELECT genus, count(*) AS cites_fin_records FROM cites_new WHERE product_group = 'fins'
     GROUP BY 1 ORDER BY 2 DESC LIMIT 5""")

,cites_term,records
0,meat,11721
1,shells,649
2,pearl,601
3,jewellery,431


,cites_term,records
0,trophies,85920
1,skins,21967
2,skulls,21063
3,rug,5600
4,tusks,2837


,genus,cites_fin_records
0,carcharhinus,2496
1,sphyrna,1826
2,isurus,1588
3,prionace,1188
4,alopias,1117


Restricted to species with a CITES listing in force (last column of the table above), every remaining gap but two falls below tenfold: trophies, shells, horn, hair, scales, cloth and `other`. They are differences in **what is traded**: LEMIS holds US imports of species CITES does not list, such as pearl-oyster and top-shell shells, deer antler, badger hair and starfish. Trophies also have a second reason:
- **Trophies:** the US is the largest importer of hunting trophies (purpose H: 17.6% of LEMIS against 3.0% of CITES, notebook 03, X10), and CITES also records trophy parts under skins and skulls.

The two gaps that stay are also differences in trade, not coding:
- **Caviar:** both vocabularies have the same term and use it for the same taxa (*Acipenser*, *Huso*); both also record some sturgeon eggs as *eggs*. The global caviar trade is simply a larger share of CITES records than of US imports.
- **Fins:** CITES fin records are of requiem sharks, hammerheads, makos, blue and thresher sharks, all listed in Appendix II in 2014 or later, when LEMIS ends; LEMIS fins are mostly of unlisted species (sea squirt, salmon).

These are stated in ALIGNING §2.1. Rule 3 holds: caviar, extracts, ivory, horn and bone remain separate groups.

## 6. A second finding: the listing in force for named species

Section 5 restricts LEMIS to species with their own CITES listing. Before that, the restriction used the appendix feature, which falls back to the genus when CITES has no record of the species (ALIGNING §2.1, as in notebooks 04 and 05). That fallback takes the **strictest** appendix in the genus, and it also applies when the LEMIS record names a species CITES has never recorded:

In [10]:
how = Q("""SELECT CASE WHEN sp_listed THEN '1 species has its own listing'
                       WHEN appendix = 'genus_listed' THEN '2 named species, no CITES record; genus listed'
                       WHEN appendix <> 'not_listed' THEN '3 no species named; genus listed'
                       ELSE '4 not listed' END AS how_the_listing_is_found,
                  count(*) AS line_items, avg(label) AS case_rate
           FROM lemis_new GROUP BY 1 ORDER BY 1""").set_index("how_the_listing_is_found")
display(how.style.format({"case_rate": "{:.2%}"}))
# the fallback of notebooks 04 and 05: the strictest appendix in force for the genus
con.execute("""CREATE OR REPLACE TABLE genus_in_force AS
    SELECT lower(coalesce(Genus, split_part(Taxon, ' ', 1))) AS genus, Year AS year,
           min(Appendix) AS appendix
    FROM cites_animals WHERE Appendix <> 'N' GROUP BY 1, 2""")
Q("""SELECT l.taxon, g.appendix AS appendix_from_genus, count(*) AS line_items
     FROM (SELECT taxon, genus, year FROM lemis_new WHERE appendix = 'genus_listed') l
     ASOF JOIN genus_in_force g ON l.genus = g.genus AND l.year >= g.year
     GROUP BY 1, 2 ORDER BY 3 DESC LIMIT 8""")

,line_items,case_rate
how_the_listing_is_found,,
1 species has its own listing,934559,2.37%
"2 named species, no CITES record; genus listed",649148,1.02%
3 no species named; genus listed,253899,7.97%
4 not listed,1724426,0.83%


,taxon,appendix_from_genus,line_items
0,mustela vison,I,88273
1,odocoileus virginianus,III,53908
2,python reticulatus,I,45312
3,tragelaphus strepsiceros,III,36647
4,cervus elaphus,I,30606
5,oryx gazella,I,30385
6,equus burchellii,I,29308
7,vulpes vulpes,II,16728


The old fallback made American mink (*Mustela vison*) Appendix I, through the black-footed ferret, and white-tailed deer Appendix III, through a listed subspecies. It also mislabels synonyms: CITES now records the reticulated python as *Malayopython reticulatus*, so the LEMIS name finds no species record and took Appendix I from the genus *Python*, although the species is in Appendix II. These line items (row 2) are enforced at almost the rate of unlisted species, not listed ones.

**Change to the rule.** The genus decides only when the record names no species (row 3, where it is informative: an 8% case rate). A named species with no CITES record gets its own value, `genus_listed`, instead of a congener's appendix. The model can learn what that value means; it is not told a listing that may be wrong. ALIGNING §2.1 is updated accordingly.

In [11]:
Q("""SELECT appendix, count(*) AS line_items, avg(label) AS case_rate FROM lemis_new
     GROUP BY 1 ORDER BY 3 DESC""").set_index("appendix").style.format({"case_rate": "{:.2%}"})

,line_items,case_rate
appendix,,
I,199483,8.90%
III,36423,6.27%
II,952552,2.35%
genus_listed,649148,1.02%
not_listed,1724426,0.83%


With the change, case rates fall in the order of protection: Appendix I, then III and II, then species CITES does not record.

## Findings

| # | Finding | Change |
|---|---|---|
| G1 | LEMIS shoes (4.3% of training line items) are mostly reptile leather; CITES records such goods as *leather products (small)* and never used *shoes* in 2015–2024 | `SHO` joins one leather group |
| G2 | LEMIS small goods, large goods and shoes are enforced at similar rates (2.0–3.1%); CITES uses *large* on 0.5% of leather records | One group, `leather_products` (trim stays in it: CITES has no term for trim) |
| G3 | LEMIS live rock is *Scleractinia*, recorded by CITES as *raw corals* | `ROC` joins `coral_raw` |
| G4 | LEMIS `VEN` is wood veneer and `FIB` plant fibre; CITES *fibres* on animal records are animal fibre | `VEN`, `FIB` and plant codes stay `other`; CITES *fibres* join `hair` |
| G5 | Of the remaining tenfold gaps, all but caviar and fins close once LEMIS is restricted to listed species; caviar uses the same term for the same taxa in both, and the sharks behind CITES fin records were listed from 2014 on | Kept as differences in trade; stated in ALIGNING §2.1 |
| G6 | The genus fallback gave named species with no CITES record the strictest appendix in their genus: 649,148 line items in the training years (866,452 in all years), including American mink as Appendix I; they are enforced at 1.02%, against 0.83% for unlisted species | Genus decides only when no species is named; otherwise `genus_listed` |

**Decision (S1.1):** the groups of section 5. Rules 1 and 3 hold, and every remaining gap of rule 2 is explained. Checks 2 and 4 of ALIGNING §2.4 (`wildguard/check_alignment.py`) now pass.